In [0]:
!pip install openai kneed hdbscan umap-learn python-igraph leidenalg python-dotenv
# pip install python-igraph leidenalg
dbutils.library.restartPython()

In [0]:

from dotenv import load_dotenv
from openai import AzureOpenAI
load_dotenv("/Workspace/Repos/morgan.wang@rci.rogers.ca/CCTS_Journey_Model/.env")
from report_generation import ConfigManager
azure_config = ConfigManager.load_azure_config()
data_config = ConfigManager.load_data_config()
processing_config = ConfigManager.load_processing_config()

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=azure_config.api_key,
    api_version=azure_config.api_version,
    azure_endpoint=azure_config.azure_endpoint,
)


In [0]:
from cluster_method import ClusteringAnalyzer
import numpy as np

# Initialize analyzer
analyzer = ClusteringAnalyzer()

# Sample embeddings (replace with your actual embeddings)
embeddings = np.random.rand(100, 30)  # 100 samples, 384 dimensions

# Apply K-Means clustering
kmeans_result = analyzer.apply_kmeans_clustering(embeddings)
print(f"K-Means found {kmeans_result['n_clusters']} clusters")

# Apply DBSCAN clustering
dbscan_result = analyzer.apply_dbscan_clustering(embeddings, min_cluster_size=10)
print(f"DBSCAN found {dbscan_result['n_clusters']} clusters with {dbscan_result['noise_percentage']:.1f}% noise")

# Apply Leiden clustering
leiden_result = analyzer.apply_leiden_clustering(embeddings, k=30, resolution_parameter=0.1)
print(f"Leiden found {leiden_result['n_clusters']} clusters with modularity {leiden_result['modularity']:.3f}")

# # Select best method (K-Means vs DBSCAN)
selection = analyzer.select_best_clustering_method( kmeans_result, dbscan_result, leiden_result)
print(f"Best method: {selection['best_method']} with score ")

In [0]:
# from ccts_theme_driver_analysis import DataProcessor
from data_processing import DataProcessor
from data_processing import EmbeddingProcessor
# from ccts_theme_driver_analysis import ThemeAnalyzer
from ccts_theme_driver_analysis import TopicAnalyzer
data_folder = "/Workspace/Users/morgan.wang@rci.rogers.ca/share_workspeace/Levels_report_generation/Journey_2512_full"
# Custom workflow
processor = DataProcessor()
embedder = EmbeddingProcessor(client)
clusterer = ClusteringAnalyzer()
topic_analyzer = TopicAnalyzer(client)

# # Process data
df = processor.process_case_journey_folder(data_folder)
texts = df['primary_complaint_issue'].tolist()

# # Generate embeddings
# embeddings = embedder.get_embeddings_in_batches(texts)

# # Cluster
# clustering_result = clusterer.apply_leiden_clustering(embeddings)

# # Extract topics
# topics = topic_analyzer.extract_topics(embeddings, clustering_result['labels'], texts)

In [0]:
df['primary_complaint_issue_clean'] = df['primary_complaint_issue'].str.split('[').str[0].str.strip()
texts = df['primary_complaint_issue_clean'].tolist()
texts = [t for t in texts if t]

In [0]:

embeddings = embedder.get_embeddings_in_batches(texts)

In [0]:
from cluster_method import ClusteringAnalyzer

clustering_result = clusterer.apply_leiden_clustering(embeddings)